# Appendix Q — Table 8 and Figures 4–6: local-projection policy paths

This notebook reproduces the empirical local-projection application in Appendix Q. It downloads public monthly macro-finance data, constructs a policy-move regressor and horizon-specific cumulative equity returns, estimates a single policy-coordinate data score, reuses that score across horizons, reports the one-sided policy-path table, and plots the policy paths for horizons 1–12.

In [ ]:

from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd()
for parent in [REPO, *REPO.parents]:
    if (parent / "src" / "genriesz" / "scorematchingriesz.py").exists():
        REPO = parent
        break
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import genriesz.scorematchingriesz as smr
import torch
import torch.nn.functional as F
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
print("device:", DEVICE)


In [ ]:
START_DATE = "1995-01-01"
END_DATE = None
HORIZONS = list(range(1, 13))
SHIFT_GRID = np.array([-1.00, -0.75, -0.50, -0.25, 0.00, 0.25, 0.50, 0.75, 1.00])
LAGS = 12
SCORE_HIDDEN_DIMS = (64, 64)
OUTCOME_HIDDEN_DIMS = (128, 128)
SCORE_STEPS = 4000
OUTCOME_EPOCHS = 2000
BATCH_SIZE = 256
DSM_SIGMA_MIN = 0.05
DSM_SIGMA_MAX = 0.5
SIGMA_EVAL = 0.05
INTEGRATION_STEPS = 64
CLIP_LOG_RATIO = 20.0
TABLE_TITLE = "Table 8: policy path table"
FIGURE_TITLE_PREFIX = "Policy path"

In [ ]:

# Public data download. This cell requires internet access.
from pandas_datareader import data as pdr
import yfinance as yf

fred_codes = {
    "fedfunds": "FEDFUNDS",
    "cpi": "CPIAUCSL",
    "unrate": "UNRATE",
    "indpro": "INDPRO",
    "vix": "VIXCLS",
}
fred = []
for name, code in fred_codes.items():
    s = pdr.DataReader(code, "fred", START_DATE, END_DATE).rename(columns={code: name})
    fred.append(s)
macro = pd.concat(fred, axis=1).resample("M").last()
macro["vix"] = pdr.DataReader("VIXCLS", "fred", START_DATE, END_DATE).rename(columns={"VIXCLS": "vix"}).resample("M").mean()["vix"]
spy = yf.download("SPY", start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)[["Close"]].rename(columns={"Close": "spy"}).resample("M").last()
raw = macro.join(spy, how="inner").dropna()
raw["policy_move"] = raw["fedfunds"].diff()
raw["ret"] = np.log(raw["spy"]).diff()
raw["inflation"] = np.log(raw["cpi"]).diff()
raw["ip_growth"] = np.log(raw["indpro"]).diff()
raw = raw.dropna()
display(raw.tail())


In [ ]:

def make_design(raw, horizons, lags):
    df = raw.copy()
    for h in horizons:
        df[f"Y_h{h}"] = sum(df["ret"].shift(-j) for j in range(1, h + 1))
    controls = ["policy_move", "ret", "inflation", "unrate", "ip_growth", "vix"]
    for col in controls:
        for lag in range(1, lags + 1):
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
    df = df.dropna()
    y_cols = [f"Y_h{h}" for h in horizons]
    z_cols = [c for c in df.columns if c.endswith(tuple(f"lag{j}" for j in range(1, lags + 1)))]
    # X = (D, Z). D is measured in percentage points. Standardize for neural training, but keep a scale map for shifts.
    X_raw = df[["policy_move"] + z_cols].to_numpy(dtype="float32")
    Y = {h: df[f"Y_h{h}"].to_numpy(dtype="float32") for h in horizons}
    mu = X_raw.mean(axis=0, keepdims=True)
    sd = X_raw.std(axis=0, keepdims=True) + 1e-8
    X = (X_raw - mu) / sd
    return df, X.astype("float32"), Y, mu.reshape(-1), sd.reshape(-1)


def shift_standardized_x(X, delta_percentage_points, sd):
    out = np.asarray(X, dtype="float32").copy()
    out[:, 0] += float(delta_percentage_points) / float(sd[0])
    return out


def newey_west_se(scores, lag):
    psi = np.asarray(scores, dtype=float) - np.mean(scores)
    T = len(psi)
    gamma0 = np.dot(psi, psi) / T
    omega = gamma0
    for ell in range(1, int(lag) + 1):
        gamma = np.dot(psi[ell:], psi[:-ell]) / T
        weight = 1.0 - ell / (int(lag) + 1.0)
        omega += 2.0 * weight * gamma
    return float(np.sqrt(max(omega, 0.0) / T))


def wald_from_scores(scores, lag):
    theta = float(np.mean(scores))
    se = newey_west_se(scores, lag)
    return theta, se, theta - 1.96 * se, theta + 1.96 * se


In [ ]:

def fit_policy_coordinate_dsm(x, *, seed=0):
    smr.set_seed(seed)
    xt = torch.tensor(np.asarray(x, dtype="float32"), device=DEVICE)
    model = smr.DataScoreDNet(xt.shape[1], hidden_dims=SCORE_HIDDEN_DIMS).to(DEVICE)
    model.treatment_index = 0
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-6)
    n = xt.shape[0]
    for _ in range(SCORE_STEPS):
        idx = torch.randint(0, n, (min(BATCH_SIZE, n),), device=DEVICE)
        xb = xt[idx]
        sigma = torch.empty((xb.shape[0], 1), device=DEVICE).uniform_(DSM_SIGMA_MIN, DSM_SIGMA_MAX)
        eps = torch.randn((xb.shape[0], 1), device=DEVICE)
        noisy = xb.clone()
        noisy[:, [0]] = noisy[:, [0]] + sigma * eps
        target = -eps / sigma
        pred = model(noisy, sigma)
        # Appendix Q states that only the policy coordinate is noised and the weighting uses lambda(sigma)=sigma^2.
        loss = ((sigma ** 2) * (pred - target) ** 2).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    model.eval()
    return model


In [ ]:

df, X, Y_BY_H, X_MEAN, X_SD = make_design(raw, HORIZONS, LAGS)
print({"T": len(df), "n_features": X.shape[1], "date_start": str(df.index.min().date()), "date_end": str(df.index.max().date())})
score_model = fit_policy_coordinate_dsm(X, seed=RANDOM_SEED)

rows = []
score_values_by_h_delta = {}
for h in HORIZONS:
    y = Y_BY_H[h]
    outcome = smr.fit_outcome_net(X, y, hidden_dims=OUTCOME_HIDDEN_DIMS, n_epochs=OUTCOME_EPOCHS, batch_size=BATCH_SIZE, seed=RANDOM_SEED + h, device=DEVICE)
    gamma_base = smr.predict_outcome(outcome, X, device=DEVICE).reshape(-1)
    residual = y - gamma_base
    for delta in SHIFT_GRID:
        X_delta = shift_standardized_x(X, delta, X_SD)
        gamma_delta = smr.predict_outcome(outcome, X_delta, device=DEVICE).reshape(-1)
        if abs(float(delta)) < 1e-12:
            ratio_minus_one = np.zeros(len(X))
        else:
            log_r = smr.log_ratio_from_data_score_shift(score_model, X, abs(float(delta)) / float(X_SD[0]), steps=INTEGRATION_STEPS, sigma_eval=SIGMA_EVAL, direction="+" if delta > 0 else "-", normalize=True, x_p_for_norm=X, device=DEVICE).reshape(-1)
            ratio_minus_one = np.exp(np.clip(log_r, -CLIP_LOG_RATIO, CLIP_LOG_RATIO)) - 1.0
        scores = gamma_delta - gamma_base + ratio_minus_one * residual
        theta, se, ci_low, ci_high = wald_from_scores(scores, lag=max(6, h))
        score_values_by_h_delta[(h, float(delta))] = scores
        rows.append({"horizon": h, "delta": float(delta), "theta": theta, "se": se, "ci_low": ci_low, "ci_high": ci_high})
path_df = pd.DataFrame(rows)
print(TABLE_TITLE)
display(path_df)


In [ ]:

# Wide display matching the layout of Table 8: one row block per delta.
wide_rows = []
for delta in SHIFT_GRID:
    sub = path_df[path_df["delta"] == float(delta)].set_index("horizon")
    wide_rows.append({"delta": float(delta), "stat": "theta", **{f"h={h}": sub.loc[h, "theta"] for h in HORIZONS}})
    wide_rows.append({"delta": float(delta), "stat": "se", **{f"h={h}": sub.loc[h, "se"] for h in HORIZONS}})
    wide_rows.append({"delta": float(delta), "stat": "CI", **{f"h={h}": f"[{sub.loc[h, 'ci_low']:.3f}, {sub.loc[h, 'ci_high']:.3f}]" for h in HORIZONS}})
wide_table = pd.DataFrame(wide_rows)
display(wide_table)


In [ ]:

def plot_horizon_group(horizons, title):
    fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
    axes = axes.ravel()
    for ax, h in zip(axes, horizons):
        sub = path_df[path_df["horizon"] == h].sort_values("delta")
        ax.plot(sub["delta"], sub["theta"], label="Estimated path from AME score")
        ax.fill_between(sub["delta"], sub["ci_low"], sub["ci_high"], alpha=0.2, label="95% CI")
        ax.axhline(0.0, linestyle="--")
        ax.set_title(f"h={h}")
        ax.set_xlabel("policy shift δ")
        ax.set_ylabel("one-sided effect θ_h(δ,0)")
    axes[0].legend(loc="best")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

plot_horizon_group([1, 2, 3, 4], f"{FIGURE_TITLE_PREFIX} with h=1,2,3,4")
plot_horizon_group([5, 6, 7, 8], f"{FIGURE_TITLE_PREFIX} with h=5,6,7,8")
plot_horizon_group([9, 10, 11, 12], f"{FIGURE_TITLE_PREFIX} with h=9,10,11,12")
